# Task W1-03 — Strategy B: 평균 회귀 일봉 백테스트

**Feature ID**: STR-B-001
**Sub-plan**: `docs/stage1-subplans/w1-03-strategy-b-daily.md`

Larry Connors 스타일 평균 회귀 전략을 일봉으로 백테스트.
- 200일 MA 추세 필터 (상승 추세에서만 역추세 매수)
- RSI(4) < 25 진입 (극단 과매도)
- RSI(4) > 50 청산 또는 5 거래일 시간 스톱 또는 -8% 하드 스톱

## 사전 지정 파라미터 (변경 금지)

| 파라미터 | 값 | 설명 |
|---------|-----|------|
| MA_PERIOD | 200 | 추세 필터 (상승장에서만 진입) |
| RSI_PERIOD | 4 | Connors 스타일 단기 RSI |
| RSI_BUY | 25 | 극단 과매도 진입 임계 |
| RSI_SELL | 50 | RSI 회복 청산 임계 |
| TIME_STOP_DAYS | 5 | 시간 스톱 (5 거래일) |
| SL_PCT | 0.08 | 하드 스톱 -8% (vectorbt sl_stop fraction) |

## 신호 로직

```
진입:
  close > ma200 (대추세 상승, EOD 신호 → 같은 바 fill)
  AND rsi(4) < 25 (극단 과매도)

청산 (하나라도 충족):
  rsi(4) > 50 (RSI 회복)
  OR entries.shift(5) (5 바 후 시간 스톱 근사)
  OR sl_stop=0.08 (-8% 하드 스톱, vectorbt 자동)
```

**시간 스톱 caveat**: `entries.shift(N)` 패턴은 "엔트리 신호 N바 후 exit" 근사.
실제 "보유 N바 후"는 아님 (vectorbt 0.28.5 무료 버전은 td_stop 미지원).
- 포지션 재진입 차단 시 약간 차이 가능
- 평균회귀 청산이 보통 2~6 바 내라 5일 시간 스톱은 거의 발동 안 함 (안전장치)

## 검증된 vectorbt 0.28.5 API (W1-02 v3 best practices 상속)

- sl_stop fraction (0.08), sl_trail=False, freq='1D'
- pf.sharpe_ratio() 메서드 호출
- ts_stop / td_stop / max_duration 사용 안 함
- ta.momentum.RSIIndicator (Wilder smoothing)

## v3 schema 상속 (W1-02 v3에서 도입)

- nested metrics dict
- sharpe_se_return_basis + T_returns=N-1 + sharpe_ci_95_normal (Lo 2002 정규 근사)
- deepest_drawdown / longest_drawdown 분리 (vectorbt drawdowns 두 metric 혼동 방지)
- edge_case_checks 동적 derive
- agent trace 저장 (.evidence/agent-reviews/)


In [1]:
import pandas as pd
import numpy as np
import vectorbt as vbt
from ta.momentum import RSIIndicator
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

from importlib.metadata import version
print(f"pandas:   {version('pandas')}")
print(f"numpy:    {version('numpy')}")
print(f"vectorbt: {version('vectorbt')}")
print(f"ta:       {version('ta')}")


pandas:   2.3.3
numpy:    2.3.5
vectorbt: 0.28.5
ta:       0.11.0


In [2]:
DATA_DIR = Path('../data')
PARQUET_NAME = 'KRW-BTC_1d_frozen_20260412.parquet'
PARQUET_PATH = DATA_DIR / PARQUET_NAME

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()

with open(DATA_DIR / 'data_hashes.txt') as f:
    expected_hashes = {}
    for line in f:
        line = line.strip()
        if line and not line.startswith('#') and ':' in line:
            k, v = line.split(': ', 1)
            expected_hashes[k] = v

DATA_HASH = sha256_file(PARQUET_PATH)
expected_hash = expected_hashes[PARQUET_NAME]
assert DATA_HASH == expected_hash, f"Data hash mismatch! expected={expected_hash}, actual={DATA_HASH}"
print(f"Data hash verified: {DATA_HASH[:16]}...")


Data hash verified: da5b5a5bd74c1be0...


In [3]:
df = pd.read_parquet(PARQUET_PATH)

assert df.index.tz is not None and str(df.index.tz) == 'UTC', f"Expected UTC, got {df.index.tz}"
assert df.index.is_monotonic_increasing, "Index not monotonic"
assert len(df) == 1927, f"Expected 1927 bars (advertised range), got {len(df)}"

close  = df['close']
high   = df['high']
low    = df['low']
volume = df['volume']

print(f"Bars: {len(df)}")
print(f"Range: {df.index[0]} ~ {df.index[-1]}")
print(f"Columns: {list(df.columns)}")
print(f"NaN total: {df.isna().sum().sum()}")


Bars: 1927
Range: 2021-01-01 00:00:00+00:00 ~ 2026-04-11 00:00:00+00:00
Columns: ['open', 'high', 'low', 'close', 'volume', 'value']
NaN total: 0


In [4]:
MA_PERIOD       = 200
RSI_PERIOD      = 4
RSI_BUY         = 25
RSI_SELL        = 50
TIME_STOP_DAYS  = 5
SL_PCT          = 0.08

INIT_CASH = 1_000_000
FEES      = 0.0005
SLIPPAGE  = 0.0005
FREQ      = '1D'


In [5]:
ma200 = close.rolling(window=MA_PERIOD).mean()
rsi = RSIIndicator(close=close, window=RSI_PERIOD).rsi()

print(f"ma200 NaN (warmup): {ma200.isna().sum()} (expected: {MA_PERIOD - 1})")
print(f"rsi NaN (warmup):   {rsi.isna().sum()} (expected: ~{RSI_PERIOD})")
print(f"rsi range:          {rsi.min():.2f} ~ {rsi.max():.2f}")


ma200 NaN (warmup): 199 (expected: 199)
rsi NaN (warmup):   3 (expected: ~4)
rsi range:          2.60 ~ 99.53


In [6]:
entries = (close > ma200) & (rsi < RSI_BUY)
entries = entries.fillna(False).astype(bool)

rsi_exits = (rsi > RSI_SELL).fillna(False).astype(bool)

# Time stop: entries.shift(N) pattern (vectorbt에 td_stop 없음)
# "엔트리 신호 N바 후 exit" 근사. 실제 "보유 N바 후"는 아님.
time_exits = entries.shift(TIME_STOP_DAYS).fillna(False).astype(bool)

exits = (rsi_exits | time_exits).astype(bool)

print(f"Total entries: {entries.sum()}")
print(f"RSI exits:     {rsi_exits.sum()}")
print(f"Time exits:    {time_exits.sum()}")
print(f"Combined exits: {exits.sum()}")

# Edge case 1: warmup zero entries
warmup_entries = int(entries.iloc[:MA_PERIOD].sum())
warmup_period_end = df.index[MA_PERIOD - 1]
print(f"\nWarmup entries (first {MA_PERIOD} bars, last warmup bar {warmup_period_end.date()}): {warmup_entries}")
assert warmup_entries == 0, f"Warmup entries must be 0, got {warmup_entries}"
print(f"  PASS")

# Edge case 2: time stop 발동 패턴 (정보용)
time_stop_active_after_warmup = int(time_exits.iloc[MA_PERIOD:].sum())
print(f"\nTime stop signals after warmup: {time_stop_active_after_warmup}")
print(f"  (정보용 — RSI 평균회귀 특성상 보통 2~6 바 내 청산되어 time stop 발동 적음)")


Total entries: 107
RSI exits:     995
Time exits:    107
Combined exits: 1059

Warmup entries (first 200 bars, last warmup bar 2021-07-19): 0
  PASS

Time stop signals after warmup: 107
  (정보용 — RSI 평균회귀 특성상 보통 2~6 바 내 청산되어 time stop 발동 적음)


/var/folders/7f/y4cccdcs08z55qw7qhk4_tdr0000gn/T/ipykernel_16543/3361490488.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  time_exits = entries.shift(TIME_STOP_DAYS).fillna(False).astype(bool)


In [7]:
pf = vbt.Portfolio.from_signals(
    close=close,
    entries=entries,
    exits=exits,
    sl_stop=SL_PCT,
    sl_trail=False,
    init_cash=INIT_CASH,
    fees=FEES,
    slippage=SLIPPAGE,
    freq=FREQ,
)
print("vectorbt portfolio created successfully (no API errors)")


vectorbt portfolio created successfully (no API errors)


In [8]:
stats = pf.stats()
print(stats)


Start                         2021-01-01 00:00:00+00:00
End                           2026-04-11 00:00:00+00:00
Period                               1927 days 00:00:00
Start Value                                   1000000.0
End Value                                1049355.998564
Total Return [%]                                 4.9356
Benchmark Return [%]                         236.162373
Max Gross Exposure [%]                            100.0
Total Fees Paid                            38259.225827
Max Drawdown [%]                              21.268289
Max Drawdown Duration                 855 days 00:00:00
Total Trades                                         39
Total Closed Trades                                  39
Total Open Trades                                     0
Open Trade PnL                                      0.0
Win Rate [%]                                  53.846154
Best Trade [%]                                10.173861
Worst Trade [%]                               -9

In [9]:
sharpe       = float(pf.sharpe_ratio())
total_return = float(pf.total_return())
max_dd       = float(pf.max_drawdown())
total_trades = int(pf.trades.count())

if total_trades > 0:
    win_rate      = float(pf.trades.win_rate())
    profit_factor = float(pf.trades.profit_factor())
else:
    win_rate = float('nan')
    profit_factor = float('nan')

# Deepest vs longest drawdown 분리 (W1-02 v3 패턴)
records = pf.drawdowns.records_readable
records['DD_pct'] = (records['Valley Value'] - records['Peak Value']) / records['Peak Value']
records['Duration_days'] = (
    pd.to_datetime(records['End Timestamp']) - pd.to_datetime(records['Peak Timestamp'])
).dt.days

if len(records) > 0:
    deepest_idx = records['DD_pct'].idxmin()
    deepest_rec = records.loc[deepest_idx]
    deepest_dd_pct = float(deepest_rec['DD_pct'])
    deepest_dd_duration_days = int(deepest_rec['Duration_days'])
    deepest_dd_status = str(deepest_rec['Status'])
    deepest_dd_recovered = (deepest_dd_status == 'Recovered')
    deepest_dd_peak = pd.to_datetime(deepest_rec['Peak Timestamp']).isoformat()
    deepest_dd_end = pd.to_datetime(deepest_rec['End Timestamp']).isoformat()

    longest_idx = records['Duration_days'].idxmax()
    longest_rec = records.loc[longest_idx]
    longest_dd_pct = float(longest_rec['DD_pct'])
    longest_dd_duration_days = int(longest_rec['Duration_days'])
    longest_dd_status = str(longest_rec['Status'])
    longest_dd_recovered = (longest_dd_status == 'Recovered')

    assert abs(deepest_dd_pct - max_dd) < 1e-9, \
        f"Deepest DD reconciliation FAILED: {deepest_dd_pct} vs {max_dd}"
    print(f"Deepest DD reconcile PASS: {deepest_dd_pct*100:.4f}% == {max_dd*100:.4f}%")
else:
    deepest_dd_pct = 0.0
    deepest_dd_duration_days = 0
    deepest_dd_status = 'None'
    deepest_dd_recovered = True
    deepest_dd_peak = None
    deepest_dd_end = None
    longest_dd_pct = 0.0
    longest_dd_duration_days = 0
    longest_dd_status = 'None'
    longest_dd_recovered = True
    print("No drawdowns recorded")

# === Realized time-stop trades 계산 ===
# time_stop_signal_count (raw mask)는 "entries.shift(5)가 True인 바 수"로,
# 실제 time-stop으로 청산된 trade 수와는 다름.
# 실제 청산이 time-stop에 의한 것인지 확인: trades.records에서 exit bar - entry bar 관계.
realized_time_stop_trades = 0
if total_trades > 0:
    tr = pf.trades.records_readable
    # Entry/Exit bar는 records_readable에 'Entry Timestamp', 'Exit Timestamp'로 있음
    tr['hold_bars'] = (pd.to_datetime(tr['Exit Timestamp']) - pd.to_datetime(tr['Entry Timestamp'])).dt.days
    # Time stop으로 청산된 것은 보유일이 정확히 TIME_STOP_DAYS (또는 그 근처) 인 경우
    # 더 정확히: rsi_exits가 먼저 발동되지 않은 상태에서 entries.shift(TIME_STOP_DAYS)으로 청산된 경우
    # 간이 추정: hold_bars == TIME_STOP_DAYS인 trade 수
    realized_time_stop_trades = int((tr['hold_bars'] == TIME_STOP_DAYS).sum())
    print(f"\nRealized time-stop trades (hold == {TIME_STOP_DAYS}d): {realized_time_stop_trades}/{total_trades}")

print(f"\nSharpe:        {sharpe:.4f}")
print(f"Total Return:  {total_return*100:.2f}%")
print(f"Max Drawdown:  {max_dd*100:.2f}%")
print(f"  Deepest DD:  {deepest_dd_pct*100:.2f}%, {deepest_dd_duration_days}d, {'recovered' if deepest_dd_recovered else 'still active'}")
print(f"  Longest DD:  {longest_dd_pct*100:.2f}%, {longest_dd_duration_days}d, {'recovered' if longest_dd_recovered else 'still active'}")
print(f"Total Trades:  {total_trades}")
if total_trades > 0:
    print(f"Win Rate:      {win_rate*100:.2f}%")
    print(f"Profit Factor: {profit_factor:.3f}")

# Sharpe SE + 95% CI (Lo 2002 정규 근사)
# 0-trade 가드: sharpe가 NaN일 수 있음
T_returns = len(close) - 1
if total_trades > 0 and not np.isnan(sharpe):
    sharpe_se = float(np.sqrt((1 + 0.5 * sharpe**2) / T_returns))
    sharpe_ci_lo = sharpe - 1.96 * sharpe_se
    sharpe_ci_hi = sharpe + 1.96 * sharpe_se
    print(f"\nSharpe SE (Lo 2002, return-basis, T_returns={T_returns}): {sharpe_se:.4f}")
    print(f"Sharpe 95% CI (정규 근사): [{sharpe_ci_lo:.4f}, {sharpe_ci_hi:.4f}]")
else:
    sharpe_se = float('nan')
    sharpe_ci_lo = float('nan')
    sharpe_ci_hi = float('nan')
    print(f"\nSharpe SE: N/A (no trades or NaN sharpe)")

print(f"\n=== Week 1 Go 기준 평가 (참고) ===")
print(f"Sharpe > 0.5: {'PASS' if sharpe > 0.5 else 'FAIL'} ({sharpe:.4f})")
print(f"MDD < 50%:    {'PASS' if abs(max_dd) < 0.5 else 'FAIL'} ({max_dd*100:.2f}%)")


Deepest DD reconcile PASS: -21.2683% == -21.2683%

Realized time-stop trades (hold == 5d): 15/39

Sharpe:        0.1362
Total Return:  4.94%
Max Drawdown:  -21.27%
  Deepest DD:  -21.27%, 715d, still active
  Longest DD:  -17.62%, 856d, recovered
Total Trades:  39
Win Rate:      53.85%
Profit Factor: 1.092

Sharpe SE (Lo 2002, return-basis, T_returns=1926): 0.0229
Sharpe 95% CI (정규 근사): [0.0913, 0.1810]

=== Week 1 Go 기준 평가 (참고) ===
Sharpe > 0.5: FAIL (0.1362)
MDD < 50%:    PASS (-21.27%)


In [10]:
out_dir = Path('../outputs')
out_dir.mkdir(exist_ok=True)

results = {
    'feature_id': 'STR-B-001',
    'task_id': 'W1-03',
    'strategy': 'B_mean_reversion_daily',
    'description': 'Larry Connors-inspired mean reversion: MA200 trend filter + RSI(4)<25 entry + RSI(4)>50 exit + 5d time stop + sl_stop 8%',
    'data_file': PARQUET_NAME,
    'data_hash': DATA_HASH,
    'data_bars': len(df),
    'data_range': [df.index[0].isoformat(), df.index[-1].isoformat()],
    'parameters': {
        'ma_period': MA_PERIOD,
        'rsi_period': RSI_PERIOD,
        'rsi_buy': RSI_BUY,
        'rsi_sell': RSI_SELL,
        'time_stop_days': TIME_STOP_DAYS,
        'sl_pct': SL_PCT,
    },
    'backtest_config': {
        'init_cash': INIT_CASH,
        'fees': FEES,
        'slippage': SLIPPAGE,
        'freq': FREQ,
    },
    'metrics': {
        'sharpe': sharpe,
        'sharpe_se_return_basis': sharpe_se,
        'sharpe_se_t_returns': T_returns,
        'sharpe_ci_95_normal': [sharpe_ci_lo, sharpe_ci_hi],
        'sharpe_ci_method': 'Lo 2002 normal approximation, return-basis',
        'total_return': total_return,
        'max_drawdown': max_dd,
        'deepest_drawdown': {
            'pct': deepest_dd_pct,
            'duration_days': deepest_dd_duration_days,
            'status': deepest_dd_status,  # 'Active' or 'Recovered'
            'recovered': deepest_dd_recovered,
            'peak_timestamp': deepest_dd_peak,
            'end_timestamp': deepest_dd_end,  # Active이면 window boundary
        },
        'longest_drawdown': {
            'pct': longest_dd_pct,
            'duration_days': longest_dd_duration_days,
            'status': longest_dd_status,
            'recovered': longest_dd_recovered,
        },
        'total_drawdown_records': len(records),
        'win_rate': win_rate if total_trades > 0 else None,
        'profit_factor': profit_factor if total_trades > 0 else None,
        'total_trades': total_trades,
    },
    'edge_case_checks': {
        'warmup_zero_entries': bool(int(entries.iloc[:MA_PERIOD].sum()) == 0),
        # raw mask bit count (실제 청산과 다름)
        'time_stop_mask_nonempty': bool(int(time_exits.iloc[MA_PERIOD:].sum()) > 0),
        'time_stop_mask_count_raw': int(time_exits.iloc[MA_PERIOD:].sum()),
        # 실제 time-stop으로 청산된 trade 수 (hold_bars == TIME_STOP_DAYS)
        'realized_time_stop_trades': realized_time_stop_trades,
        'deepest_dd_reconciles_with_max_drawdown': bool(abs(deepest_dd_pct - max_dd) < 1e-9) if total_trades > 0 else True,
    },
    'go_criteria_eval': {
        'sharpe_gt_0.5': sharpe > 0.5,
        'mdd_lt_50pct': abs(max_dd) < 0.5,
    },
    'generated_at': datetime.now(timezone.utc).isoformat(),
}

result_path = out_dir / 'strategy_b_daily.json'
with open(result_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"Saved: {result_path}")
print(json.dumps(results['metrics'], indent=2, default=str))


Saved: ../outputs/strategy_b_daily.json
{
  "sharpe": 0.13615418374262483,
  "sharpe_se_return_basis": 0.02289155640309841,
  "sharpe_se_t_returns": 1926,
  "sharpe_ci_95_normal": [
    0.09128673319255196,
    0.18102163429269771
  ],
  "sharpe_ci_method": "Lo 2002 normal approximation, return-basis",
  "total_return": 0.04935599856437929,
  "max_drawdown": -0.21268288552328618,
  "deepest_drawdown": {
    "pct": -0.21268288552328599,
    "duration_days": 715,
    "status": "Active",
    "recovered": false,
    "peak_timestamp": "2024-04-26T00:00:00+00:00",
    "end_timestamp": "2026-04-11T00:00:00+00:00"
  },
  "longest_drawdown": {
    "pct": -0.17617121531070978,
    "duration_days": 856,
    "status": "Recovered",
    "recovered": true
  },
  "total_drawdown_records": 4,
  "win_rate": 0.5384615384615384,
  "profit_factor": 1.0915409254532455,
  "total_trades": 39
}


In [11]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

equity = pf.value()
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True, gridspec_kw={'height_ratios': [3, 1]})

axes[0].plot(equity.index, equity.values, label='Strategy B equity', linewidth=1.2, color='steelblue')
axes[0].axhline(INIT_CASH, color='gray', linestyle='--', alpha=0.5, label='Initial cash')
axes[0].set_title('Strategy B (Mean Reversion Daily) — Equity Curve')
axes[0].set_ylabel('Portfolio Value (KRW)')
axes[0].legend(loc='upper left')
axes[0].grid(True, alpha=0.3)

drawdown = (equity / equity.cummax() - 1) * 100
axes[1].fill_between(drawdown.index, drawdown.values, 0, color='red', alpha=0.3)
axes[1].set_title('Drawdown (%)')
axes[1].set_ylabel('Drawdown %')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(out_dir / 'strategy_b_equity.png', dpi=100, bbox_inches='tight')
plt.close()
print(f"Equity curve saved: {out_dir / 'strategy_b_equity.png'}")


Equity curve saved: ../outputs/strategy_b_equity.png


In [12]:
print("=" * 60)
print("W1-03 Strategy B Verification Summary")
print("=" * 60)
print(f"Feature ID:    STR-B-001")
print(f"Data:          {PARQUET_NAME}")
print(f"Data hash:     {DATA_HASH[:16]}...")
print(f"Bars:          {len(df)} (range {df.index[0].date()} ~ {df.index[-1].date()})")
print()
print("Pre-registered parameters:")
print(f"  MA={MA_PERIOD}, RSI({RSI_PERIOD}) buy<{RSI_BUY} sell>{RSI_SELL}, time_stop={TIME_STOP_DAYS}d, SL={SL_PCT}")
print()
print("Backtest results:")
print(f"  Sharpe:        {sharpe:.4f} (95% CI [{sharpe_ci_lo:.4f}, {sharpe_ci_hi:.4f}], 정규 근사)")
print(f"  Total Return:  {total_return*100:.2f}%")
print(f"  Max Drawdown:  {max_dd*100:.2f}%")
print(f"    Deepest DD:  {deepest_dd_pct*100:.2f}%, {deepest_dd_duration_days}d, {'recovered' if deepest_dd_recovered else 'still active'}")
print(f"    Longest DD:  {longest_dd_pct*100:.2f}%, {longest_dd_duration_days}d, {'recovered' if longest_dd_recovered else 'still active'}")
print(f"  Total Trades:  {total_trades}")
if total_trades > 0:
    print(f"  Win Rate:      {win_rate*100:.2f}%")
    print(f"  Profit Factor: {profit_factor:.3f}")
print()
print("Edge case checks:")
print(f"  Warmup zero entries: PASS")
print(f"  Time stop signals after warmup: {int(time_exits.iloc[MA_PERIOD:].sum())}")
print()
print("Go criteria (W1-06에서 종합 평가):")
print(f"  Sharpe > 0.5: {'PASS' if sharpe > 0.5 else 'FAIL'} ({sharpe:.4f})")
print(f"  MDD < 50%:    {'PASS' if abs(max_dd) < 0.5 else 'FAIL'} ({max_dd*100:.2f}%)")


W1-03 Strategy B Verification Summary
Feature ID:    STR-B-001
Data:          KRW-BTC_1d_frozen_20260412.parquet
Data hash:     da5b5a5bd74c1be0...
Bars:          1927 (range 2021-01-01 ~ 2026-04-11)

Pre-registered parameters:
  MA=200, RSI(4) buy<25 sell>50, time_stop=5d, SL=0.08

Backtest results:
  Sharpe:        0.1362 (95% CI [0.0913, 0.1810], 정규 근사)
  Total Return:  4.94%
  Max Drawdown:  -21.27%
    Deepest DD:  -21.27%, 715d, still active
    Longest DD:  -17.62%, 856d, recovered
  Total Trades:  39
  Win Rate:      53.85%
  Profit Factor: 1.092

Edge case checks:
  Warmup zero entries: PASS
  Time stop signals after warmup: 107

Go criteria (W1-06에서 종합 평가):
  Sharpe > 0.5: FAIL (0.1362)
  MDD < 50%:    PASS (-21.27%)
